# 第 5 章 · 多智能体与工作流

> **你已会**：Agent 参数、工具、`output_key`、事件流。
>
> **读完后你能**：① 判断一个问题该不该拆成多 Agent；② 用 `Sequential / Parallel / Loop` 写出确定性流水线；③ 用 `sub_agents` 做 LLM 路由并观察 `transfer_to_agent`；④ 说清"转移"和"调用"的区别；⑤ 验证回调在父子 Agent 间的传播。
>
> **本章不碰**：动态工作流（第 8 章）、恢复与人工确认（第 9 章）。

**怎么读**

- **主线（约 60 分钟）**：§1 总图 → §2 Sequential → §3 Parallel → §5 委派 → §6 怎么选。
- **深读**：§4 Loop 的退出机制、§5.2 转移观察、§5.3 回调传播。
- 本章代码会真实调用模型，每次运行输出可能略有不同。**先看事件流的作者顺序，再看文本内容。**

---

## 1. 多智能体：先想清楚"为什么拆"

单 Agent 的天花板很具体：

- prompt 越写越长，不同规则互相干扰；
- 一个大任务无法同时做多件事；
- 所有工具都暴露给同一个模型，权限过大。

但**多 Agent 不是默认答案**。它只在至少满足一条时才划算：

| 条件 | 例子 |
|---|---|
| 职责差异大，写在一个 prompt 里会互相干扰 | 教务 vs IT 支持 |
| 任务可以并行，且结果能合并 | 三个视角同时分析 |
| 需要隔离上下文或权限 | 退款 Agent 不应看到用户闲聊 |
| 需要不同的模型参数或工具集 | 创意写作 vs 事实查询 |

如果只是"一个 prompt 写长一点"，拆成多 Agent 通常只会增加成本和调试难度。

### 1.1 母模型 ③：谁决定下一步

第 1 章说过，多智能体的核心问题是**控制权**。ADK 给两种答案：

```mermaid
flowchart TB
    subgraph P1["范式一：模型决定路线"]
        R["🧭 父 Agent<br/>读 description 做路由"]
        R -.转移.-> SA["子 Agent A"]
        R -.转移.-> SB["子 Agent B"]
    end
    subgraph P2["范式二：开发者决定路线"]
        SEQ["⛓️ Sequential<br/>串行"]
        PAR["🌿 Parallel<br/>并行"]
        LOOP["🔁 Loop<br/>循环"]
    end
    style R fill:#e8f0fe,stroke:#4285f4,stroke-width:2px
    style SEQ fill:#e6f4ea,stroke:#34a853
    style PAR fill:#fef7e0,stroke:#fbbc04
    style LOOP fill:#fce8e6,stroke:#ea4335
```

| 范式 | 谁决定下一步 | 可预测性 | 适合 |
|---|---|---|---|
| LLM 委派 | 模型 | 中 | 开放意图、客服分流 |
| Sequential | 你 | 高 | 写-改-审流水线 |
| Parallel | 你 | 高 | 多视角分析、批处理 |
| Loop | 你定规则，模型决定何时停 | 中高 | 迭代打磨 |

> 📌 **一句话**：委派买的是"灵活"，工作流买的是"可预测"。两者可以混用，§6 会给判断标准。

<details>
<summary><b>版本说明：工作流 Agent 已标记 deprecated</b></summary>

ADK 2.6+ 已把 `SequentialAgent / ParallelAgent / LoopAgent` 标记为 deprecated，官方正在演进为 `google.adk.workflow.Workflow` 的图编排（第 8、9 章）。本章仍用它们教学，原因是：概念会原样平移到新 API，而且当前版本完全可用。运行时看到 `DeprecationWarning` 属于预期。

</details>

先准备本章的运行辅助函数。它会显示每个事件的**作者**，方便观察 Agent 之间的接力：

In [1]:
import os
assert os.environ.get("DEEPSEEK_API_KEY"), "请先设置 DEEPSEEK_API_KEY"

from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
from dotenv import load_dotenv

load_dotenv()

APP, USER = "adk_ch04", "student"
DS = lambda: LiteLlm(model="deepseek/deepseek-chat")  # 每个 Agent 独立模型实例

async def run_team(root_agent, query, session_id="demo", show_state_keys=None):
    """运行一个（可能是多智能体的）Agent 系统，打印接力过程，返回 session。"""
    ss = InMemorySessionService()
    await ss.create_session(app_name=APP, user_id=USER, session_id=session_id)
    runner = Runner(agent=root_agent, app_name=APP, session_service=ss)
    msg = types.Content(role="user", parts=[types.Part(text=query)])
    print(f"🧑 {query}\n" + "═" * 60)
    async for ev in runner.run_async(user_id=USER, session_id=session_id, new_message=msg):
        if not (ev.content and ev.content.parts):
            continue
        for p in ev.content.parts:
            if p.function_call:
                print(f"  ⚡ [{ev.author}] 调用 {p.function_call.name}({dict(p.function_call.args)})")
            elif p.function_response:
                print(f"  📦 [{ev.author}] {p.function_response.name} 返回（略）")
            elif p.text:
                tag = "🏁 最终" if ev.is_final_response() else "🗣️ 中间"
                print(f"  {tag} [{ev.author}] {p.text[:120]}")
    session = await ss.get_session(app_name=APP, user_id=USER, session_id=session_id)
    if show_state_keys:
        print("═" * 60, "\n🗂️ Session 状态快照：")
        for k in show_state_keys:
            v = session.state.get(k, "（无）")
            print(f"  {k} = {str(v)[:100]}")
    return session

print("✅ 环境就绪")


✅ 环境就绪


---

## 2. 范式二 · SequentialAgent：串行流水线

**场景**：内容生产流水线——`大纲 → 初稿 → 润色`，每一步由专职 Agent 完成，上一步的产出通过 `output_key` 存入状态，下一步用 `{key}` 插值读取。

```mermaid
flowchart LR
    U["主题输入"] --> A["1️⃣ 大纲师<br/>output_key=outline"]
    A -->|"state['outline']"| B["2️⃣ 撰稿人<br/>读{outline}<br/>output_key=draft"]
    B -->|"state['draft']"| C["3️⃣ 润色师<br/>读{draft}<br/>output_key=final"]
    C --> O["成品文案"]
    style A fill:#e8f0fe,stroke:#4285f4
    style B fill:#e6f4ea,stroke:#34a853
    style C fill:#fef7e0,stroke:#fbbc04
```

这就是第 3 章 `output_key` + `{state插值}` 的实战组合：


In [2]:
from google.adk.agents import SequentialAgent

outliner = Agent(
    name="outliner", model=DS(),
    instruction="你是写作大纲师。根据用户主题列出 3 个要点的大纲，每点一句话。只输出大纲。",
    description="生成文章大纲",
    output_key="outline",
)

writer = Agent(
    name="writer", model=DS(),
    instruction="你是撰稿人。根据下面的大纲写一段 150 字左右的短文：\n{outline}\n只输出正文。",
    description="根据大纲写初稿",
    output_key="draft",
)

polisher = Agent(
    name="polisher", model=DS(),
    instruction="你是润色师。把下面的初稿改得更生动（保持字数，加入一个比喻）：\n{draft}\n只输出润色后的成稿。",
    description="润色初稿成成品",
    output_key="final",
)

pipeline = SequentialAgent(
    name="writing_pipeline",
    sub_agents=[outliner, writer, polisher],  # 按列表顺序依次执行
)

session = await run_team(pipeline, "主题：程序员为什么需要学习写作", session_id="seq",
                         show_state_keys=["outline", "final"])


🧑 主题：程序员为什么需要学习写作
════════════════════════════════════════════════════════════


C:\Users\37245\AppData\Local\Temp\ipykernel_41548\3322429214.py:24: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  pipeline = SequentialAgent(


  🏁 最终 [outliner] 1. 写作能帮助程序员将复杂的逻辑与技术方案梳理成清晰文档，提升团队协作与知识传递效率。
2. 良好的写作能力有助于程序员准确表达需求、设计思路与技术决策，减少沟通误解和返工成本。
3. 写作是程序员沉淀经验、建立个人影响力与推动职业长期发
  🏁 最终 [writer] 写作能帮助程序员将复杂的逻辑与技术方案梳理成清晰文档，提升团队协作与知识传递效率。良好的写作能力有助于程序员准确表达需求、设计思路与技术决策，减少沟通误解和返工成本。更重要的是，写作是程序员沉淀经验、建立个人影响力与推动职业长期发展的重要工
  🏁 最终 [polisher] 写作能帮助程序员将复杂的逻辑与技术方案梳理成清晰文档，提升团队协作与知识传递效率。良好的写作能力有助于程序员准确表达需求、设计思路与技术决策，减少沟通误解和返工成本。更重要的是，写作是程序员沉淀经验、建立个人影响力与推动职业长期发展的重要工
════════════════════════════════════════════════════════════ 
🗂️ Session 状态快照：
  outline = 1. 写作能帮助程序员将复杂的逻辑与技术方案梳理成清晰文档，提升团队协作与知识传递效率。
2. 良好的写作能力有助于程序员准确表达需求、设计思路与技术决策，减少沟通误解和返工成本。
3. 写作是程序员
  final = 写作能帮助程序员将复杂的逻辑与技术方案梳理成清晰文档，提升团队协作与知识传递效率。良好的写作能力有助于程序员准确表达需求、设计思路与技术决策，减少沟通误解和返工成本。更重要的是，写作是程序员沉淀经验、


> 🔍 观察输出中每个事件的作者：`outliner → writer → polisher` 严格按序接力，状态在三人之间无缝传递。**整个过程没有任何 LLM 路由决策**——这就是"确定性编排"的可靠感。

---

## 3. 范式二 · ParallelAgent：并行扇出

**场景**：商业分析——同一命题，**技术、市场、风险**三个视角同时开分析会，最后由汇总员合并结论。

```mermaid
flowchart TD
    U["命题输入"] --> P{"ParallelAgent<br/>同时启动"}
    P --> T["🔬 技术分析师<br/>output_key=tech"]
    P --> M["📈 市场分析师<br/>output_key=market"]
    P --> K["⚠️ 风险分析师<br/>output_key=risk"]
    T --> S["🧾 汇总员<br/>读三个 key 合并"]
    M --> S
    K --> S
    style P fill:#fef7e0,stroke:#fbbc04,stroke-width:2px
```

> ⚠️ **关键规则**：并行分支共享同一份 Session 状态，所以每个分支必须写**不同的** `output_key`，否则会互相覆盖。

并行 + 串行可以**自由嵌套**——这正是 ADK 工作流的组合美学：


In [3]:
from google.adk.agents import ParallelAgent

def make_analyst(name, angle, key):
    return Agent(
        name=name, model=DS(),
        instruction=f"你是{angle}分析师。只从{angle}视角分析用户给的命题，80 字以内，直入主题。",
        description=f"{angle}视角分析",
        output_key=key,
    )

brainstorm = ParallelAgent(
    name="brainstorm",
    sub_agents=[
        make_analyst("tech_analyst", "技术可行性", "tech"),
        make_analyst("market_analyst", "市场前景", "market"),
        make_analyst("risk_analyst", "潜在风险", "risk"),
    ],
)

merger = Agent(
    name="merger", model=DS(),
    instruction="""你是首席顾问。综合三位分析师的意见给出一段总结与一句行动建议：
【技术】{tech}
【市场】{market}
【风险】{risk}""",
    description="汇总多视角分析",
    output_key="report",
)

# Sequential 套 Parallel：先并行扇出，再串行汇总
think_tank = SequentialAgent(name="think_tank", sub_agents=[brainstorm, merger])

session = await run_team(think_tank, "命题：在高校推广 AI 编程助教", session_id="par",
                         show_state_keys=["report"])


🧑 命题：在高校推广 AI 编程助教
════════════════════════════════════════════════════════════


C:\Users\37245\AppData\Local\Temp\ipykernel_41548\1853445323.py:11: DeprecationWarning: ParallelAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  brainstorm = ParallelAgent(
C:\Users\37245\AppData\Local\Temp\ipykernel_41548\1853445323.py:31: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  think_tank = SequentialAgent(name="think_tank", sub_agents=[brainstorm, merger])


  🏁 最终 [risk_analyst] 推广AI编程助教存在数据隐私泄露、算法偏见误导学生、过度依赖削弱基础能力等风险。
  🏁 最终 [market_analyst] 高校需求明确，AI编程助教可降教学成本、提学习效率，付费意愿强，易规模化推广，市场前景乐观。
  🏁 最终 [tech_analyst] 技术可行：LLM代码生成成熟，高校算力与API可支撑，需解决数据隐私、模型幻觉及课程适配。
  🏁 最终 [merger] 三位分析师从技术、市场、风险三个维度对“在高校推广 AI 编程助教”进行了评估，核心结论如下：技术上，LLM 代码生成已趋成熟，高校算力与 API 具备支撑能力，可行性较高，但需解决数据隐私、模型幻觉和课程适配问题；市场上，高校需求明确，该
════════════════════════════════════════════════════════════ 
🗂️ Session 状态快照：
  report = 三位分析师从技术、市场、风险三个维度对“在高校推广 AI 编程助教”进行了评估，核心结论如下：技术上，LLM 代码生成已趋成熟，高校算力与 API 具备支撑能力，可行性较高，但需解决数据隐私、模型幻觉


---

## 4. 范式二 · LoopAgent：迭代打磨循环

**场景**：文案打磨——`写手` 与 `毒舌评审` 反复过招，直到评审满意为止。循环的**退出机制**有两种：

1. `max_iterations`：兜底上限，防止死循环；
2. **主动退出**：某个 Agent 调用一个设置 `tool_context.actions.escalate = True` 的工具，循环立即终止。

```mermaid
flowchart LR
    W["✍️ 写手<br/>写/改文案"] --> C["🧐 评审<br/>打分"]
    C -->|"不满意：给意见"| W
    C -->|"满意：调用 exit_loop"| E(["✅ 退出循环"])
    L["LoopAgent<br/>max_iterations=4 兜底"] -.包裹.-> W
    style L fill:#fce8e6,stroke:#ea4335,stroke-width:2px
```


In [4]:
from google.adk.agents import LoopAgent
from google.adk.tools import ToolContext

def exit_loop(tool_context: ToolContext) -> dict:
    """当文案质量已经达标时调用此工具，立即结束迭代循环。"""
    tool_context.actions.escalate = True   # 向 LoopAgent 发出"收工"信号
    return {"status": "评审通过，循环结束"}

copywriter = Agent(
    name="copywriter", model=DS(),
    instruction="你是广告写手。根据主题写一句 slogan；如果会话中已有评审意见，必须针对意见修改后再提交。只输出 slogan 本身。",
    description="撰写/修改 slogan",
    output_key="slogan",
)

critic = Agent(
    name="critic", model=DS(),
    instruction="""你是毒舌评审。审查当前 slogan：{slogan}
- 如果朗朗上口且有记忆点 → 直接调用 exit_loop 工具，不要输出多余文字；
- 否则 → 用一句话指出问题（存入你的回复），供写手下一轮修改。""",
    description="评审 slogan 并决定是否收工",
    tools=[exit_loop],
)

polish_loop = LoopAgent(
    name="polish_loop",
    sub_agents=[copywriter, critic],  # 每一轮：先写后评
    max_iterations=4,                 # 兜底：最多 4 轮
)

session = await run_team(polish_loop, "主题：无糖气泡水，主打『0 负担的快乐』", session_id="loop",
                         show_state_keys=["slogan"])


🧑 主题：无糖气泡水，主打『0 负担的快乐』
════════════════════════════════════════════════════════════


C:\Users\37245\AppData\Local\Temp\ipykernel_41548\1897030076.py:25: DeprecationWarning: LoopAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  polish_loop = LoopAgent(


  🏁 最终 [copywriter] 0糖0卡，快乐冒泡不打烊。
  🗣️ 中间 [critic] "冒泡不打烊"是个亮点，但前半句"0糖0卡"只是干巴巴的成分声明，和后半句的爽感没有咬合成一句完整的记忆钩子。
  ⚡ [critic] 调用 exit_loop({})
  📦 [critic] exit_loop 返回（略）
  🏁 最终 [critic] 评审通过，循环结束。
════════════════════════════════════════════════════════════ 
🗂️ Session 状态快照：
  slogan = 0糖0卡，快乐冒泡不打烊。


> 🔍 你可能看到 1~4 轮不等的"写 → 评"往复——循环次数由**评审的判断**动态决定。这就是 LoopAgent 的"自我修正"能力。

---

## 5. LLM 驱动委派：智能路由

工作流 Agent 的路线是**你画好的**；委派模式把路由决策交给**模型**：

- 父 Agent 声明 `sub_agents=[...]` 后，框架自动注入一个 `transfer_to_agent` 工具；
- 父 Agent 阅读各子 Agent 的 `description`，决定"这单给谁"；
- 转移后，**子 Agent 接管对话**：它能看到完整历史，用户后续消息也直接由它处理。

**场景**：校园服务台——前台 Agent 只分流，不答题。

```mermaid
flowchart TD
    U["🧑 学生提问"] --> F["🧭 前台 Agent<br/>只做分流"]
    F -->|"transfer_to_agent<br/>读 description 决策"| A["📚 教务专家<br/>课表/考试/学分"]
    F -->|"transfer_to_agent"| B["💻 IT 支持<br/>账号/网络/设备"]
    A --> U
    B --> U
    style F fill:#e8f0fe,stroke:#4285f4,stroke-width:2px
```

**先猜：如果用户一句话里同时问"课表"和"校园网"，会发生几次转移？**

In [5]:
academic = Agent(
    name="academic_affairs", model=DS(),
    instruction="你是教务处专家，用中文简洁回答课表、考试、学分相关问题。",
    description="负责教务问题：课程表、考试安排、学分与成绩政策。",  # ← 路由依据！
)

it_support = Agent(
    name="it_support", model=DS(),
    instruction="你是校园 IT 支持，用中文给出账号、网络、设备问题的排查步骤，分点说明。",
    description="负责 IT 问题：校园网、账号登录、邮箱与电脑故障。",   # ← 路由依据！
)

front_desk = Agent(
    name="front_desk", model=DS(),
    instruction="你是服务台前台，自己不回答问题，必须根据问题类型把对话转交给对应专家。",
    description="校园服务台总机",
    sub_agents=[academic, it_support],   # ← 声明可委派的对象
)

await run_team(front_desk, "你好，我连不上校园网了，而且想问问下周的高数考试安排。", session_id="route")


🧑 你好，我连不上校园网了，而且想问问下周的高数考试安排。
════════════════════════════════════════════════════════════
  🗣️ 中间 [front_desk] I'll transfer you to the appropriate specialists for each of these.
  ⚡ [front_desk] 调用 transfer_to_agent({'agent_name': 'it_support'})
  📦 [front_desk] transfer_to_agent 返回（略）
  🗣️ 中间 [it_support] 你好！校园网的问题我来帮你排查。考试安排属于教务，我稍后帮你转过去。

**校园网连接排查步骤：**

1. **确认是否已连接正确的 Wi-Fi 信号**
   - 学生区一般连 `Campus-Student`，宿舍区连 `Campus
  ⚡ [it_support] 调用 transfer_to_agent({'agent_name': 'academic_affairs'})
  📦 [it_support] transfer_to_agent 返回（略）
  🏁 最终 [academic_affairs] 你好！高数考试安排我来解答。不过具体考试时间、考场需要按你的课程和班级查询，我这边没有你的选课信息，可以这样获取：

1. **教务系统查询**：登录教务管理平台 →「考试安排」或「我的考试」，可看到课程名称、考试时间、考场和座位号。
2.


Session(id='route', app_name='adk_ch04', user_id='student', state={}, events=[Event(model_version=None, content=Content(
  parts=[
    Part(
      text='你好，我连不上校园网了，而且想问问下周的高数考试安排。'
    ),
  ],
  role='user'
), grounding_metadata=None, partial=None, turn_complete=None, turn_complete_reason=None, finish_reason=None, error_code=None, error_message=None, interrupted=None, custom_metadata=None, usage_metadata=None, live_session_resumption_update=None, live_session_id=None, go_away=None, voice_activity=None, input_transcription=None, output_transcription=None, avg_logprobs=None, logprobs_result=None, cache_metadata=None, citation_metadata=None, interaction_id=None, environment_id=None, invocation_id='e-0c810c17-abca-489d-86d5-bd933b8a0a17', author='user', actions=EventActions(skip_summarization=None, state_delta={}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction=None, end_of_agent=None, agent_state=None, rewin

> 🔍 观察事件流里的 `transfer_to_agent` 调用——模型自己读了 description、自己做了（甚至可能两次）路由决策。

> 📌 **委派模式的天然限制**：转移是"一对一接管"。一个复合需求要么被拆成多轮，要么在子 Agent 内部继续委派。设计时不要指望一次路由解决所有问题。

### 5.1 `AgentTool` 和 `sub_agents` 到底差在哪

第 4 章已经用过 `AgentTool`，这里把两者放在一起对比：

| 维度 | `AgentTool` | `sub_agents` 转移 |
|---|---|---|
| 关系 | 调用-返回 | 移交-接管 |
| 控制权 | 主 Agent 始终掌握 | 交给子 Agent |
| 上下文 | 子 Agent 只处理这次调用 | 子 Agent 看到完整历史 |
| 后续消息 | 仍由主 Agent 处理 | 直接由子 Agent 处理 |
| 适合 | 借用专业能力（翻译、计算） | 这件事从此归你负责 |

> 📌 判断方法：问自己"这件事做完后，谁继续和用户对话？" 是主 Agent，就用 `AgentTool`；是子 Agent，就用 `sub_agents`。

---

### 5.2 把 `transfer_to_agent` 看清楚

上一段输出里已经出现了转移，但只是"看个热闹"。真正的转移是框架自动注入的一个**工具调用**，参数里带着目标 Agent 名。

下面单独跑一次路由，只收集 `transfer_to_agent` 调用：

**先猜：前台 Agent 会不会先回答一句，再转交？转移参数里带的是 `description` 还是 `name`？**


In [6]:
async def collect_routing(query: str, session_id: str) -> list[dict]:
    service = InMemorySessionService()
    await service.create_session(app_name=APP, user_id=USER, session_id=session_id)
    runner = Runner(agent=front_desk, app_name=APP, session_service=service)
    message = types.Content(role="user", parts=[types.Part(text=query)])
    transfers = []
    async for event in runner.run_async(
        user_id=USER, session_id=session_id, new_message=message
    ):
        for part in (event.content.parts if event.content else []):
            if part.function_call and part.function_call.name == "transfer_to_agent":
                transfers.append(dict(part.function_call.args))
    return transfers

transfers = await collect_routing("我连不上校园网了，怎么排查？", "routing_demo")
print("转移记录：", transfers)

assert transfers, "没有发生 transfer_to_agent"
assert transfers[0]["agent_name"] == "it_support", transfers
print("✅ 路由依据是子 Agent 的 name；description 只用于让模型做判断")

转移记录： [{'agent_name': 'it_support'}]
✅ 路由依据是子 Agent 的 name；description 只用于让模型做判断



### 5.3 回调在父子 Agent 之间怎么传播

一个常见误解：给父 Agent 挂了回调，子 Agent 的模型调用也会自动被它拦截。

**事实是：每个 Agent 只触发自己身上的回调。** 顺序由编排结构决定，而不是自动继承。

下面用 `SequentialAgent` 串两个 Agent，各挂一个记录型回调：

**先猜：最终记录的顺序是 `a → b`，还是 `b → a`？**


In [7]:
callback_order: list[str] = []

def mark(name: str):
    def before_model(callback_context, llm_request):
        callback_order.append(name)
        return None
    return before_model

step_a = Agent(
    name="step_a", model=DS(),
    instruction="用一句话说明什么是 Agent。只输出一句话。",
    description="第一步",
    before_model_callback=mark("a"),
)
step_b = Agent(
    name="step_b", model=DS(),
    instruction="把上一条回答改写成更短的一句。只输出改写结果。",
    description="第二步",
    before_model_callback=mark("b"),
)

callback_chain = SequentialAgent(name="callback_chain", sub_agents=[step_a, step_b])
await run_team(callback_chain, "介绍一下 ADK。", session_id="callback_chain")

print("回调记录：", callback_order)
assert callback_order == ["a", "b"], callback_order
print("✅ 每个 Agent 只触发自己的回调；顺序由编排决定")

🧑 介绍一下 ADK。
════════════════════════════════════════════════════════════


C:\Users\37245\AppData\Local\Temp\ipykernel_41548\1042373819.py:22: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  callback_chain = SequentialAgent(name="callback_chain", sub_agents=[step_a, step_b])


  🏁 最终 [step_a] ADK（Agent Development Kit）是谷歌推出的开源智能体开发框架，用于帮助开发者构建、编排和部署基于大模型的 AI Agent。
  🏁 最终 [step_b] ADK 是谷歌推出的开源智能体开发框架，用于构建、编排和部署基于大模型的 AI Agent。
回调记录： ['a', 'b']
✅ 每个 Agent 只触发自己的回调；顺序由编排决定



### 6.1 什么时候该嵌套，什么时候该拍平

嵌套能表达复杂结构，但也会让事件路径变长、调试变难。一个实用判断：

| 情况 | 建议 |
|---|---|
| 子流程有自己的输入输出边界，可单独测试 | 嵌套 |
| 只是三步串行，没有复用需求 | 拍平，直接 `SequentialAgent` |
| 并行分支内部还需要串行加工 | 嵌套 `Sequential(Parallel(...))` |
| 嵌套超过三层 | 先停下来，考虑拆成独立工程或动态工作流 |

> 📌 **可观察性优先**：嵌套层级越深，`node_info.path` 越长，出错时越难定位。能用两个扁平工作流讲清的结构，不要用四层嵌套。

## 6. 两种范式怎么选

| 考量 | LLM 委派 | 工作流编排 |
|---|---|---|
| 路线是否已知 | 未知、开放意图 | 已知、步骤明确 |
| 可靠性 | 中（模型可能选错） | 高（按图执行） |
| token 开销 | 高（每步都要决策） | 低 |
| 延迟 | 高（多一次模型决策） | 低；Parallel 还能提速 |
| 调试难度 | 高（要看模型为什么这样选） | 低（顺序固定） |
| 典型场景 | 客服分流、开放问答 | 内容生产、数据处理 |

> 💡 **真实系统几乎都是混合的**：顶层用 LLM 委派做意图路由，每个子 Agent 内部用 Sequential / Loop 做确定性加工。ADK 允许任意嵌套——工作流 Agent 可以作为 `sub_agents` 被委派，LLM Agent 也可以作为工作流的一环。

**一个实用判断顺序**：

1. 步骤能不能提前写死？能 → 先用工作流；
2. 只有"选哪条路"不确定？→ 在那一层用委派；
3. 委派后还需要主 Agent 继续处理？→ 用 `AgentTool`，不要用转移。

---

<details>
<summary><b>深入：与 LangGraph 对照</b></summary>

| ADK | LangGraph 等价物 | 差异 |
|---|---|---|
| `SequentialAgent` | 直线边链 `A→B→C` | LangGraph 更啰嗦，但可在任意节点插逻辑 |
| `ParallelAgent` | 扇出边 + Reducer 合并状态 | LangGraph 需自己设计合并策略 |
| `LoopAgent` + escalate | 条件边构成回边 | 本质相同：循环即回边 |
| `transfer_to_agent` | Command/handoff + supervisor 模式 | ADK 内建；LangGraph 需自建 |
| `output_key` + `{state}` | 节点返回 dict 更新 State | ADK 声明式；LangGraph 显式 |

**一句话**：ADK 把常见模式预制成了积木；LangGraph 给图纸和零件。ADK 新一代 `Workflow` API 正在转向图编排，两者在"图"这个点上越走越近。

</details>

---

## 📌 用五句话复述本章

1. **多 Agent 不是默认答案**，只在职责差异大、可并行、需隔离权限时才拆。
2. **两种范式的本质区别是"谁决定下一步"**：模型委派 vs 你写死。
3. **Sequential 靠 `output_key` 接力；Parallel 必须用不同 key，且完成顺序不保证。**
4. **Loop 的 `max_iterations` 是兜底，不是成功条件**；必须区分"达标退出"和"预算耗尽"。
5. **`AgentTool` 是调用-返回，`sub_agents` 是移交-接管**；判断标准是"之后谁继续和用户对话"。

> ➡️ 下一章：[06-会话-状态与记忆](06-会话-状态与记忆.ipynb) —— 深入 ADK 的三层记忆体系。

---

## 🧪 本章练习

### 1. 串行内容流水线（基础）

用 `SequentialAgent` 构建"需求摘要 → 技术方案 → 风险审查"。

- [ ] 每一步使用独立 `output_key`
- [ ] 后一步显式消费前一步结果
- [ ] 检查 Session State，确认中间产物可追踪
- [ ] 为某一步输出为空设计降级处理

### 2. 并行调研与结果汇总（进阶）

- [ ] 用 `ParallelAgent` 并行完成用户需求、技术可行性、合规风险三路分析
- [ ] 汇总 Agent 生成最终建议
- [ ] 三个分支写入不同的 `output_key`
- [ ] 记录串行与并行版本的总耗时
- [ ] 说明共享状态键冲突时会发生什么

### 3. 有上限的迭代打磨（进阶）

用 `LoopAgent` 实现"撰写 → 评分 → 修改"。

- [ ] 评分达到阈值时通过 `escalate` 退出
- [ ] 设置最大迭代次数
- [ ] 未达标时输出最后版本及失败原因
- [ ] 覆盖三条路径：一次通过、改进后通过、达到上限仍失败

### 4. 混合式客服系统（工程综合）

- [ ] 顶层用 LLM 委派，路由到退款、物流、技术支持
- [ ] 退款子 Agent 内部使用"资格检查 → 人工确认占位 → 执行"的确定性流水线
- [ ] 复用第 3 章的模糊请求测试集，统计路由准确率
- [ ] 保证任何退款动作都不能只凭一次模型判断直接执行

### 5. 把 ADK 设计翻译成状态图（跨框架）

选择练习 2 或 4，画出等价的 LangGraph 节点、边、条件边和 State 字段。

- [ ] 标出 `sub_agents` 的"移交-接管"在图上的表现
- [ ] 标出 `AgentTool` 的"调用-返回"在图上的表现
- [ ] 判断哪一种更容易限制 token 成本和最大步数，并说明理由